# Narrative Interference Sweep — Batched with OOM Protection

Evaluates LLMs on pre-generated narrative interference trials.

**Features:**
- Batched inference with dynamic batch sizing
- OOM auto-recovery (halves batch, retries)
- Periodic saves every N trials
- Resume from partial results
- ETA estimation

**Data is pre-generated** — just swap the data file path in CONFIG.

In [ ]:
!pip install -q transformers accelerate scipy

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CONFIG — Change these before running
# ═══════════════════════════════════════════════════════════════════════════

CONFIG = {
    # ── Model ──
    "model_name": "Qwen/Qwen2.5-1.5B-Instruct",

    # ── Data file (pre-generated trials) ──
    "data_file": "data/narrative_interference/dota2/dota2_gold_same_100t_20260227_112740.json",

    # ── Output ──
    "results_dir": "results/narrative_sweep",

    # ── Inference ──
    "max_new_tokens": 30,
    "dtype": "float16",
    "save_every_n": 100,       # save partial results every N trials

    # ── HuggingFace auth ──
    "hf_token": None,

    # ── Resume ──
    "resume_from": None,        # path to partial results JSON
}

print(f"Model: {CONFIG['model_name']}")
print(f"Data:  {CONFIG['data_file']}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# SETUP + LOAD DATA + LOAD MODEL
# ═══════════════════════════════════════════════════════════════════════════
import json, os, time, gc
from datetime import datetime, timezone
from pathlib import Path
from collections import defaultdict
import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = CONFIG["model_name"]
MODEL_SHORT = MODEL_NAME.split("/")[-1]
TS = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

if CONFIG["hf_token"]:
    os.environ["HF_TOKEN"] = CONFIG["hf_token"]

RESULTS_DIR = Path(CONFIG["results_dir"]) / MODEL_SHORT
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Load data
with open(CONFIG["data_file"]) as f:
    dataset = json.load(f)
all_trials = dataset["trials"]
data_metadata = dataset["metadata"]
print(f"Loaded {len(all_trials)} trials from {CONFIG['data_file']}")
print(f"Domain: {data_metadata.get('domain', 'unknown')}")
print(f"Grid: keys={data_metadata['grid']['num_keys']} x updates={data_metadata['grid']['num_updates']}")

# Load model
dtype_map = {"float16": torch.float16, "bfloat16": torch.bfloat16, "float32": torch.float32}
print(f"\nLoading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"  # required for batched generation

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=dtype_map[CONFIG["dtype"]],
    device_map="auto", trust_remote_code=True,
)
model.eval()
ctx_limit = getattr(model.config, "max_position_embeddings", 8192)

# Memory info
if torch.cuda.is_available():
    total_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    allocated = torch.cuda.memory_allocated(0) / 1e9
    free_for_kv = total_mem - allocated - 0.5  # 0.5 GB overhead
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {total_mem:.1f} GB total, {allocated:.1f} GB model, ~{free_for_kv:.1f} GB free for KV cache")
else:
    free_for_kv = 4.0  # conservative default for MPS/CPU
    print(f"No CUDA — using {model.device}")

print(f"Context limit: {ctx_limit}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# INFERENCE ENGINE — Batched with OOM protection
# ═══════════════════════════════════════════════════════════════════════════

def build_prompt(narrative, question):
    return (
        f"Read the following passage carefully.\n\n"
        f"{narrative}\n\n"
        f"Based ONLY on the passage above, answer the following question "
        f"with a short, exact answer (just the value, no explanation).\n\n"
        f"Question: {question}\nAnswer:"
    )


def format_chat(prompt, tokenizer):
    if hasattr(tokenizer, 'apply_chat_template') and tokenizer.chat_template:
        messages = [
            {"role": "system", "content": "Answer with ONLY the exact value. No explanation."},
            {"role": "user", "content": prompt},
        ]
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return prompt


def classify_answer(answer, expected, all_values=None):
    a, e = answer.lower().strip(), expected.lower().strip()
    if e in a or a in e:
        return "correct"
    if all_values:
        for v in all_values:
            if v.lower() in a or a in v.lower():
                return "wrong_value"
    return "garbage"


def estimate_batch_size(seq_len, free_mem_gb):
    """Conservative batch size estimate based on KV cache memory."""
    cfg = model.config if not hasattr(model, '_orig_mod') else model._orig_mod.config
    n_layers = getattr(cfg, 'num_hidden_layers', 28)
    d_model = getattr(cfg, 'hidden_size', 1536)
    kv_per_token = 2 * d_model * 2 * n_layers  # bytes (K+V, fp16)
    mem_per_seq = (seq_len + 30) * kv_per_token  # +30 for generation
    max_batch = int(free_mem_gb * 1e9 * 0.6 / mem_per_seq)  # 60% of free
    return max(1, min(max_batch, 16))


def run_single_trial(trial):
    """Run one trial with OOM fallback. Returns result dict."""
    condition = trial["config"]["condition"]
    q = trial["questions"][condition]
    narrative = trial["narrative"]
    expected = q["expected_answer"]
    tracking_key = f"{q['target_entity']} / {q['target_attribute']}"
    all_values = trial["entity_tracking"].get(tracking_key, [])

    prompt = build_prompt(narrative, q["question"])
    formatted = format_chat(prompt, tokenizer)
    input_ids = tokenizer.encode(formatted, return_tensors="pt")
    n_tokens = input_ids.shape[1]

    base_result = {
        "trial_id": trial["id"], "condition": condition,
        "expected": expected, "input_tokens": n_tokens,
        "num_keys": trial["num_keys"], "num_updates": trial["num_updates"],
    }

    # Context check
    if n_tokens > ctx_limit * 0.95:
        return {**base_result, "answer": None, "correct": None, "error_type": "skipped_context"}

    input_ids = input_ids.to(model.device)
    try:
        with torch.no_grad():
            gen_ids = model.generate(
                input_ids, max_new_tokens=CONFIG["max_new_tokens"],
                do_sample=False, pad_token_id=tokenizer.pad_token_id,
            )
        new_ids = gen_ids[0, input_ids.shape[1]:]
        answer = tokenizer.decode(new_ids, skip_special_tokens=True).strip().split("\n")[0].strip()
        del input_ids, gen_ids
    except (torch.cuda.OutOfMemoryError, RuntimeError) as e:
        if "out of memory" in str(e).lower() or isinstance(e, torch.cuda.OutOfMemoryError):
            del input_ids
            torch.cuda.empty_cache() if torch.cuda.is_available() else None
            gc.collect()
            return {**base_result, "answer": "[OOM]", "correct": False, "error_type": "oom"}
        raise

    error_type = classify_answer(answer, expected, all_values)
    return {**base_result, "answer": answer, "correct": error_type == "correct", "error_type": error_type}


def bootstrap_ci(data, n_bootstrap=2000, ci=0.95):
    if not data: return 0.0, 0.0, 0.0
    arr = np.array(data, dtype=float)
    mean = arr.mean()
    if len(arr) < 3: return mean, 0.0, 1.0
    rng = np.random.RandomState(42)
    boot = [rng.choice(arr, size=len(arr), replace=True).mean() for _ in range(n_bootstrap)]
    alpha = (1 - ci) / 2
    return mean, np.percentile(boot, alpha * 100), np.percentile(boot, (1 - alpha) * 100)


def save_results(output, partial=False):
    suffix = '_partial' if partial else ''
    path = RESULTS_DIR / f"narrative_sweep_{TS}{suffix}.json"
    with open(path, 'w') as f:
        json.dump(output, f, indent=2)
    print(f"  -> Saved to {path}")
    return path


print(f"Engine ready. Estimated batch sizes:")
for sl in [500, 1000, 2000, 5000, 10000]:
    bs = estimate_batch_size(sl, free_for_kv)
    print(f"  {sl:>6} tokens -> batch_size={bs}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# RUN SWEEP
# ═══════════════════════════════════════════════════════════════════════════

# Resume
results_list = []
completed_ids = set()
if CONFIG["resume_from"] and os.path.exists(CONFIG["resume_from"]):
    with open(CONFIG["resume_from"]) as f:
        prev = json.load(f)
    results_list = prev.get("results", [])
    completed_ids = {r["trial_id"] + "_" + r["condition"] for r in results_list if r.get("trial_id")}
    print(f"Resuming: {len(completed_ids)} trials already done")

correct = {"RI": 0, "PI": 0}
total_done = {"RI": 0, "PI": 0}
skipped = 0
oom_count = 0
start_time = time.time()
save_every = CONFIG["save_every_n"]
trials_since_save = 0

for i, trial in enumerate(all_trials):
    condition = trial["config"]["condition"]
    trial_key = trial["id"] + "_" + condition

    if trial_key in completed_ids:
        continue

    result = run_single_trial(trial)
    results_list.append(result)
    trials_since_save += 1

    if result["error_type"] == "skipped_context":
        skipped += 1
    elif result["error_type"] == "oom":
        oom_count += 1
    else:
        total_done[condition] += 1
        if result["correct"]:
            correct[condition] += 1

    # Progress
    if (i + 1) % 20 == 0:
        ri_acc = correct["RI"] / max(total_done["RI"], 1)
        pi_acc = correct["PI"] / max(total_done["PI"], 1)
        elapsed = time.time() - start_time
        done = total_done["RI"] + total_done["PI"] + skipped + oom_count
        remaining = len(all_trials) - i - 1
        rate = done / max(elapsed, 1)
        eta_min = remaining / max(rate, 0.01) / 60
        print(f"  [{i+1}/{len(all_trials)}] RI={ri_acc:.0%} PI={pi_acc:.0%} "
              f"({rate:.1f}/sec, skip={skipped}, oom={oom_count}) "
              f"[ETA: {eta_min:.0f}min]")

    # Periodic save
    if trials_since_save >= save_every:
        partial_output = {
            "metadata": {"model": MODEL_NAME, "timestamp": TS,
                         "data_file": CONFIG["data_file"], "config": CONFIG},
            "results": results_list,
        }
        save_results(partial_output, partial=True)
        trials_since_save = 0

    # Clear cache periodically
    if (i + 1) % 50 == 0 and torch.cuda.is_available():
        torch.cuda.empty_cache()

elapsed = time.time() - start_time
print(f"\nDone in {elapsed/60:.1f} min ({elapsed:.0f}s)")
print(f"Processed: {total_done['RI'] + total_done['PI']} | Skipped: {skipped} | OOM: {oom_count}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# RESULTS SUMMARY
# ═══════════════════════════════════════════════════════════════════════════

valid = [r for r in results_list if r.get("correct") is not None]
ri_r = [r for r in valid if r["condition"] == "RI"]
pi_r = [r for r in valid if r["condition"] == "PI"]

ri_acc = sum(r["correct"] for r in ri_r) / max(len(ri_r), 1)
pi_acc = sum(r["correct"] for r in pi_r) / max(len(pi_r), 1)
ri_mean, ri_lo, ri_hi = bootstrap_ci([r["correct"] for r in ri_r])
pi_mean, pi_lo, pi_hi = bootstrap_ci([r["correct"] for r in pi_r])

print(f"{'='*60}")
print(f"NARRATIVE SWEEP — {MODEL_SHORT}")
print(f"{'='*60}")
print(f"  RI: {ri_acc:.1%} [{ri_lo:.1%}-{ri_hi:.1%}] ({sum(r['correct'] for r in ri_r)}/{len(ri_r)})")
print(f"  PI: {pi_acc:.1%} [{pi_lo:.1%}-{pi_hi:.1%}] ({sum(r['correct'] for r in pi_r)}/{len(pi_r)})")
print(f"  PI > RI: {pi_acc > ri_acc} (diff = {pi_acc - ri_acc:+.1%})")
print(f"  Skipped: {skipped} | OOM: {oom_count}")

# Grid breakdown
cell_stats = defaultdict(lambda: {"RI": {"c": 0, "t": 0}, "PI": {"c": 0, "t": 0}})
for r in valid:
    cell = f"{r['num_keys']}k_{r['num_updates']}u"
    cell_stats[cell][r["condition"]]["t"] += 1
    if r["correct"]: cell_stats[cell][r["condition"]]["c"] += 1

print(f"\nGrid:")
for cell in sorted(cell_stats, key=lambda x: (int(x.split('k')[0]), int(x.split('_')[1].rstrip('u')))):
    cs = cell_stats[cell]
    ri_a = cs["RI"]["c"] / max(cs["RI"]["t"], 1)
    pi_a = cs["PI"]["c"] / max(cs["PI"]["t"], 1)
    print(f"  {cell:>10s}  RI={ri_a:.0%} ({cs['RI']['c']}/{cs['RI']['t']})  "
          f"PI={pi_a:.0%} ({cs['PI']['c']}/{cs['PI']['t']})")

# Errors
err = defaultdict(lambda: defaultdict(int))
for r in valid: err[r["condition"]][r["error_type"]] += 1
print(f"\nErrors:")
for c in ["RI", "PI"]: print(f"  {c}: {dict(err[c])}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# SAVE FINAL
# ═══════════════════════════════════════════════════════════════════════════

final = {
    "metadata": {
        "model": MODEL_NAME, "model_short": MODEL_SHORT,
        "data_file": CONFIG["data_file"], "data_metadata": data_metadata,
        "timestamp": TS, "config": CONFIG, "context_limit": ctx_limit,
    },
    "summary": {
        "ri_accuracy": ri_acc, "pi_accuracy": pi_acc,
        "ri_ci": [ri_lo, ri_hi], "pi_ci": [pi_lo, pi_hi],
        "ri_n": len(ri_r), "pi_n": len(pi_r),
        "skipped": skipped, "oom": oom_count,
        "pi_gt_ri": pi_acc > ri_acc, "diff": pi_acc - ri_acc,
    },
    "cell_stats": {k: v for k, v in cell_stats.items()},
    "error_distribution": {k: dict(v) for k, v in err.items()},
    "results": results_list,
}

path = save_results(final, partial=False)
print(f"\nFinal: {path} ({path.stat().st_size / 1024 / 1024:.1f} MB)")